# Cleaning & Validation Layer (Bronze → Silver)

Consolidated, modular notebook that cleans and validates **five** datasets and writes
validated **parquet** files to `SILVER_DIR` (default `data/silver/`, per
`documentation/cleaning_and_validation_layer.md`).

**Databricks mode:** reads from bronze Delta tables produced by `01_ingestion_layer.ipynb`
via `spark.read.table(...)`.

| Dataset | Source bronze table(s) | Reporting grain | Output file |
|---|---|---|---|
| Police crime | `bronze_<force>_crime` × 4 forces | `(crime_id, lsoa_code)` + ASB pass-through | `<force>_crime_clean.parquet` (per force) |
| ADI claimant | `bronze_adi_claimant` | `lsoa_code × year` | `claimant_clean.parquet` |
| ADI crime | `bronze_adi_crime` | `lsoa_code × year` | `adi_crime_clean.parquet` |
| ADI health | `bronze_adi_health` | `lsoa_code × year` | `health_clean.parquet` |
| House prices | `bronze_price_paid` | Transaction-level | `houseprices_clean.parquet` |
| Postcode → LSOA | `bronze_postcode_lookup` | Postcode (lookup) | `postcode_clean.parquet` |

Each dataset section follows the same structure:
1. **Load** — read from the bronze Delta table
2. **Clean** — handle missing values, duplicates, inconsistent categories
3. **Validate** — row counts before/after each transformation, null checks, duplicate checks at the intended grain, category sanity checks
4. **Write parquet** to `SILVER_DIR`

Forces in `POLICE_FORCES` whose bronze table has not yet been ingested are skipped gracefully (printed warning), so the notebook is safe to run before every force CSV is uploaded.

## 1. Imports & Configuration

In [0]:
import os
import pandas as pd
from functools import reduce
from pyspark.sql import functions as F, DataFrame
from pyspark.sql.window import Window

# ── Catalog / schema (must match 01_ingestion_layer.ipynb) ─────────────────
CATALOG = "crime_data"
SCHEMA  = "bronze"

# Output root. Default per documentation/cleaning_and_validation_layer.md.
# On Databricks override to f"/Volumes/{CATALOG}/silver/outputs" if desired.
SILVER_DIR = f"/Volumes/{CATALOG}/silver/outputs"

# Forces to ingest — same list as 01_ingestion_layer.ipynb Section 1.
POLICE_FORCES = [
    "cambridgeshire",
    "avon_and_somerset",
    "merseyside",
    "nottinghamshire",
]

# ── House-prices constants (per documentation/houseprices_preprocess.md) ───
# Operational columns kept after cleaning (transaction_id carried for dedup, dropped after)
HP_OUTPUT_COLUMNS = [
    "transaction_id", "date_of_transfer", "postcode",
    "property_type", "ppd_category_type", "price",
]

RESIDENTIAL = {"D", "S", "T", "F"}

# Standard Home Office crime categories (data.police.uk taxonomy).
CRIME_TYPES = {
    "Anti-social behaviour",
    "Bicycle theft",
    "Burglary",
    "Criminal damage and arson",
    "Drugs",
    "Other crime",
    "Other theft",
    "Possession of weapons",
    "Public order",
    "Robbery",
    "Shoplifting",
    "Theft from the person",
    "Vehicle crime",
    "Violence and sexual offences",
}

# ── Crime outcome priority (lower number == kept on duplicate) ─────────────
# Used by dedupe_crime() in Section 4. Unknown outcomes fall back to 99.
OUTCOME_PRIORITY = {
    "Offender sent to prison": 1,
    "Suspect charged as part of another case": 2,
    "Offender given a caution": 3,
    "Offender given a drugs possession warning": 4,
    "Offender given community sentence": 5,
    "Offender given a fine": 6,
    "Local resolution": 7,
    "Action to be taken by another organisation": 8,
    "Unable to prosecute suspect": 9,
    "Formal action is not in the public interest": 10,
    "Investigation complete; no suspect identified": 11,
    "Status update unavailable": 12,
    "Under investigation": 13,
    "Awaiting court outcome": 14,
    "Court result unavailable": 15,
    "Court case unable to proceed": 16,
    "Defendant found not guilty": 17,
    "Defendant sent to Crown Court": 18,
    "Not Recorded": 99,
}

## 2. Reusable Utilities

Pure functions shared across all dataset sections.

In [0]:
def standardise_columns(df, rename_map=None):
    """Strip, lowercase, spaces->underscores; then apply optional rename_map."""
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )
    if rename_map:
        df = df.rename(columns=rename_map)
    return df


def standardise_postcode(series):
    """Uppercase, strip, collapse internal whitespace to single space (ONS pcds format)."""
    return (
        series.str.upper()
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )


def read_bronze(table):
    """Read a bronze Delta table and return a pandas DataFrame."""
    return spark.read.table(f"{CATALOG}.{SCHEMA}.{table}").toPandas()


def read_bronze_spark(table):
    """Read a bronze Delta table as a Spark DataFrame (no toPandas)."""
    return spark.read.table(f"{CATALOG}.{SCHEMA}.{table}")


def dedupe_aggregate(df, group_cols, agg_map):
    """Group by group_cols and aggregate with agg_map to resolve duplicates at a grain."""
    return df.groupby(group_cols, as_index=False).agg(agg_map)


def write_parquet(df, path):
    """Write df to parquet to Databricks Volume path and log shape."""
    # Convert Spark DataFrame to pandas if needed
    if hasattr(df, 'toPandas'):
        df = df.toPandas()
    
    # Use Spark to write parquet — works with Volume paths
    sdf = spark.createDataFrame(df)
    sdf.write \
        .mode("overwrite") \
        .parquet(path)
    
    print(f"✅ Written: {path}  shape={df.shape}")

## 3. Validation Helpers

Explicit checks used at every transformation stage across all datasets.

In [0]:
def assess_quality(df, name=""):
    """Print a data-quality summary: shape, dtype, null counts/%, unique count per column."""
    label = f" [{name}]" if name else ""
    print(f"\n=== Quality Assessment{label} ===")
    print(f"Shape: {df.shape}")
    summary = pd.DataFrame({
        "dtype":    df.dtypes.astype(str),
        "nulls":    df.isnull().sum(),
        "null_%":   (df.isnull().mean() * 100).round(2),
        "n_unique": df.nunique(),
    })
    display(summary)


def check_row_counts(label, before, after):
    """Print before -> after row counts with dropped count and retention %."""
    dropped = before - after
    pct_kept = round(after / before * 100, 1) if before else 0.0
    print(f"{label}: {before:,} -> {after:,}  (dropped {dropped:,}, kept {pct_kept}%)")


def check_duplicates(df, subset, name=""):
    """Assert zero duplicates on subset columns at the intended reporting grain."""
    n = df[list(subset)].duplicated().sum()
    label = f" [{name}]" if name else ""
    status = "PASS" if n == 0 else f"FAIL -- {n:,} duplicates"
    print(f"Duplicate check{label} on {list(subset)}: {status}")
    assert n == 0, f"Expected 0 duplicates on {list(subset)}, found {n}"


def check_categories(df, col, expected):
    """Print value_counts for col; assert all values are within expected set."""
    print(f"\nValue counts -- {col}:")
    display(df[col].value_counts())
    unexpected = set(df[col].dropna().unique()) - set(expected)
    status = "PASS" if not unexpected else f"FAIL -- unexpected: {unexpected}"
    print(f"Category check -- {col}: {status}")
    assert not unexpected, f"Unexpected values in '{col}': {unexpected}"

---
## Dataset 1: Police Crime (per force)

- **Sources:** bronze Delta tables `bronze_<force>_crime` for each force in `POLICE_FORCES`
- **Reporting grain:** `(crime_id, lsoa_code)` for crime-id rows; ASB rows (null `crime_id`) pass through untouched by design
- **Outputs:** `{SILVER_DIR}/<force>_crime_clean.parquet` — one parquet per force (e.g. `nottinghamshire_crime_clean.parquet`, `merseyside_crime_clean.parquet`)

**Cleaning strategy** (one consistent set of rules applied per force):
1. Drop ingestion provenance columns (`_source_file`, `_ingest_ts`)
2. Drop `context` (100% null) and `falls_within` (duplicate of `reported_by`)
3. Drop rows with null `lsoa_code` or `lsoa_name` (cross-boundary records)
4. Trim whitespace on `crime_type`, `last_outcome_category`, `location`
5. Parse `month` (yyyy-MM) → `year`, `month_num`, `quarter`, `month_name`
6. Derive `has_location` boolean from `longitude` / `latitude` (keeping coords)
7. Replace null `last_outcome_category` with `"Not Recorded"`
8. Keep null `crime_id` as null (identifies ASB rows)
9. Add constant `force_name = <force>`
10. Outcome-priority dedup on `(crime_id, lsoa_code)` for non-null `crime_id` rows, applied **per force** (no cross-force union — `crime_id` is force-scoped, so per-force dedup is equivalent to dedup-after-union)
11. Write one parquet per force

Forces whose bronze table is missing are skipped with a printed warning.

### 4a. Crime helper functions

In [0]:
def bronze_table_exists(force: str) -> bool:
    """Return True iff bronze_<force>_crime exists in the catalog.

    Allows the per-force loop to skip forces whose CSVs have not yet been
    ingested (mirrors the same graceful behaviour in 01_ingestion_layer.ipynb).
    """
    return spark.catalog.tableExists(f"{CATALOG}.{SCHEMA}.bronze_{force}_crime")


def clean_force_crime(force: str) -> DataFrame:
    """Read bronze_<force>_crime and apply the unified crime cleaning rules.

    Steps (see Section 4 markdown for the full rationale):
      - drop provenance / 100%-null / duplicate columns
      - drop rows with null lsoa_code / lsoa_name
      - trim text fields
      - parse month -> year / month_num / quarter / month_name
      - derive has_location boolean
      - fill null last_outcome_category with 'Not Recorded'
      - keep null crime_id as null (ASB sentinel)
      - tag rows with force_name = <force>
    Returns a Spark DataFrame ready for unioning with other forces.
    """
    df = read_bronze_spark(f"bronze_{force}_crime")

    # 1-2. Drop provenance and useless columns (only those present).
    drop_cols = [c for c in ["_source_file", "_ingest_ts", "context", "falls_within"]
                 if c in df.columns]
    if drop_cols:
        df = df.drop(*drop_cols)

    # 3. Drop rows with null lsoa_code / lsoa_name.
    df = df.filter(F.col("lsoa_code").isNotNull() & F.col("lsoa_name").isNotNull())

    # 4. Trim text columns where present.
    for col in ["crime_type", "last_outcome_category", "location", "reported_by"]:
        if col in df.columns:
            df = df.withColumn(col, F.trim(F.col(col)))

    # 5. Parse `month` (yyyy-MM) into year / month_num / quarter / month_name.
    month_date = F.to_date(F.col("month"), "yyyy-MM")
    df = (
        df.withColumn("year",        F.year(month_date))
          .withColumn("month_num",   F.month(month_date))
          .withColumn("quarter",     F.quarter(month_date))
          .withColumn("month_name",  F.date_format(month_date, "MMMM"))
          .drop("month")
    )

    # 6. has_location boolean derived from lat/long.
    df = df.withColumn(
        "has_location",
        F.col("longitude").isNotNull() & F.col("latitude").isNotNull(),
    )

    # 7. Fill null last_outcome_category with 'Not Recorded'.
    if "last_outcome_category" in df.columns:
        df = df.withColumn(
            "last_outcome_category",
            F.when(F.col("last_outcome_category").isNull(), F.lit("Not Recorded"))
             .otherwise(F.col("last_outcome_category")),
        )

    # 8. Keep crime_id null where empty / blank (identifies ASB rows naturally).
    df = df.withColumn(
        "crime_id",
        F.when(
            F.col("crime_id").isNull() | (F.trim(F.col("crime_id")) == ""),
            F.lit(None).cast("string"),
        ).otherwise(F.trim(F.col("crime_id"))),
    )

    # 9. Tag with force name so the unioned frame is queryable by force.
    df = df.withColumn("force_name", F.lit(force))

    return df


def dedupe_crime(df: DataFrame) -> DataFrame:
    """Outcome-priority dedup on (crime_id, lsoa_code) for non-null crime_id rows.

    Records with a non-null crime_id may appear in multiple monthly files when
    outcomes are revised. We keep the row with the lowest OUTCOME_PRIORITY value
    (i.e. the most resolved outcome), tie-breaking by most recent (year, month_num)
    so a Jan 2021 record beats a Dec 2020 record on the same outcome.

    ASB rows (null crime_id) have no stable key and are passed through untouched.
    """
    # Build outcome priority column via mapping.
    mapping_expr = F.create_map(
        *[F.lit(x) for kv in OUTCOME_PRIORITY.items() for x in kv]
    )
    with_priority = df.withColumn(
        "_outcome_priority",
        F.coalesce(mapping_expr[F.col("last_outcome_category")], F.lit(99)),
    )

    with_id = with_priority.filter(F.col("crime_id").isNotNull())
    null_id = with_priority.filter(F.col("crime_id").isNull())

    window = (
        Window.partitionBy("crime_id", "lsoa_code")
              .orderBy(
                  F.col("_outcome_priority").asc(),
                  F.col("year").desc(),
                  F.col("month_num").desc(),
              )
    )
    deduped = (
        with_id.withColumn("_rn", F.row_number().over(window))
               .filter(F.col("_rn") == 1)
               .drop("_rn")
    )

    reunited = deduped.unionByName(null_id).drop("_outcome_priority")
    return reunited

### 4b. Per-force clean — populate a `{force → Spark DataFrame}` dict

In [0]:
cleaned_per_force = {}
for force in POLICE_FORCES:
    if not bronze_table_exists(force):
        print(f"WARN: bronze_{force}_crime not found - skipping")
        continue
    cleaned = clean_force_crime(force)
    n = cleaned.count()
    print(f"  {force}: {n:,} rows after per-force clean")
    cleaned_per_force[force] = cleaned

assert cleaned_per_force, "No force bronze tables found - cannot proceed."
print(f"\nForces cleaned: {len(cleaned_per_force)} / {len(POLICE_FORCES)}")

### 4c. Per-force outcome-priority deduplication

`crime_id` is force-scoped (each force issues its own), so deduplicating per force is equivalent to dedup-after-union. ASB rows (null `crime_id`) pass through untouched inside `dedupe_crime`.

In [0]:
for force in list(cleaned_per_force.keys()):
    df = cleaned_per_force[force]
    before = df.count()
    deduped = dedupe_crime(df)
    after = deduped.count()
    check_row_counts(f"{force} outcome-priority dedup", before, after)
    cleaned_per_force[force] = deduped

### 4d. Collect to pandas + validate (per force)

In [0]:
# Stable column order for the silver schema (applied per force).
crime_schema = [
    "crime_id", "force_name", "reported_by",
    "year", "month_num", "quarter", "month_name",
    "longitude", "latitude", "has_location", "location",
    "lsoa_code", "lsoa_name", "crime_type", "last_outcome_category",
]

crime_clean_per_force = {}
for force, df_spark in cleaned_per_force.items():
    df = df_spark.toPandas()
    df = df[[c for c in crime_schema if c in df.columns]]

    print(f"\n=== {force}_crime_clean ===")
    assess_quality(df, f"{force}_crime_clean")

    # Duplicate check at the intended grain (non-null crime_id subset).
    crime_with_id = df[df["crime_id"].notna()]
    check_duplicates(crime_with_id, ["crime_id", "lsoa_code"], f"{force}_crime_clean (crime_id not null)")

    # Anti-social behaviour rows are intentionally preserved (null crime_id).
    asb_count = df["crime_id"].isna().sum()
    assert asb_count > 0, f"{force}: expected non-zero ASB rows (null crime_id) - dedup may have dropped them"
    print(f"ASB rows preserved (null crime_id): {asb_count:,}")

    print("\nTop crime types:")
    display(df["crime_type"].value_counts().head(20))

    # Assert crime_type values are within the standard Home Office taxonomy.
    check_categories(df, "crime_type", CRIME_TYPES)

    crime_clean_per_force[force] = df

### 4e. Write per-force parquets

In [0]:
for force, df in crime_clean_per_force.items():
    write_parquet(df, f"{SILVER_DIR}/{force}_crime_clean.parquet")

---
## Dataset 2: ADI (Area Deprivation Index)

- **Sources:** bronze Delta tables `bronze_adi_claimant`, `bronze_adi_crime`, `bronze_adi_health`
- **Reporting grain:** `lsoa_code × year`
- **Outputs:** `{SILVER_DIR}/{claimant,adi_crime,health}_clean.parquet` (per metric)
- **Columns (post-clean, per metric):** `lsoa_code`, `lsoa_name`, `pop`, `year`, `claimant_rate` *(or)* `total_crime_rate` *(or)* `total_prevalence_rate`

> Note: the ADI crime output is named `adi_crime_clean.parquet` (not `crime_clean.parquet`)
> so it doesn't collide with the police crime output written in Section 4.
> `03_feature_engineering_and_transformation` must read this filename.

The `year` column is already present on each bronze ADI table (derived from the `ADI_<YYYY>` folder name by the ingestion layer).

### 5a. ADI-specific functions

In [0]:
def prepare_claimant(df):
    """Standardise column names and keep the columns needed for deduplication."""
    df = standardise_columns(df, rename_map={"area_code": "lsoa_code", "area_name": "lsoa_name"})
    df = df[["lsoa_code", "lsoa_name", "pop",
             "claimant_rate", "claimant_count", "year"]].copy()
    df["pop"] = pd.to_numeric(df["pop"], errors="coerce")
    df["claimant_rate"] = pd.to_numeric(df["claimant_rate"], errors="coerce")
    df["claimant_count"] = pd.to_numeric(df["claimant_count"], errors="coerce")
    return df


def prepare_adi_crime(df):
    """Sum every *_rate column -> total_crime_rate; keep lsoa_code, total_crime_rate, year."""
    df = standardise_columns(df, rename_map={"area_code": "lsoa_code"})
    df = df.copy()
    rate_cols = [c for c in df.columns if c.endswith("_rate")]
    for col in rate_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["total_crime_rate"] = df[rate_cols].sum(axis=1)
    return df[["lsoa_code", "total_crime_rate", "year"]]


def prepare_health(df):
    """Sum every *_prevalence_rate column -> total_prevalence_rate; keep lsoa_code, total_prevalence_rate, year."""
    df = standardise_columns(df, rename_map={"area_code": "lsoa_code"})
    df = df.copy()
    rate_cols = [c for c in df.columns if c.endswith("_prevalence_rate")]
    for col in rate_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["total_prevalence_rate"] = df[rate_cols].sum(axis=1)
    return df[["lsoa_code", "total_prevalence_rate", "year"]]

### 5b. Load from bronze Delta tables

In [0]:
claimant  = prepare_claimant(read_bronze("bronze_adi_claimant"))
adi_crime = prepare_adi_crime(read_bronze("bronze_adi_crime"))
health    = prepare_health(read_bronze("bronze_adi_health"))

print(f"claimant : {len(claimant):,} rows")
print(f"adi_crime: {len(adi_crime):,} rows")
print(f"health   : {len(health):,} rows")

### 5c. Quality assessment

In [0]:
assess_quality(claimant,  "claimant")
assess_quality(adi_crime, "adi_crime")
assess_quality(health,    "health")

### 5d. Year coverage validation

In [0]:
print("Rows per year -- claimant:")
display(claimant["year"].value_counts().sort_index())
print("\nRows per year -- adi_crime:")
display(adi_crime["year"].value_counts().sort_index())
print("\nRows per year -- health:")
display(health["year"].value_counts().sort_index())

### 5e. Deduplication -- collapse to one row per (lsoa_code, year)

In [0]:
# Claimant: average numeric cols within (lsoa_code, year); keep first lsoa_name
claimant_clean = dedupe_aggregate(
    claimant,
    group_cols=["lsoa_code", "year"],
    agg_map={
        "lsoa_name":      "first",
        "pop":            "mean",
        "claimant_rate":  "mean",
        "claimant_count": "mean",
    },
)
claimant_clean["claimant_rate"] = claimant_clean["claimant_rate"].round(2)
claimant_clean["pop"]           = claimant_clean["pop"].round().astype("Int64")

check_row_counts("claimant dedup", len(claimant), len(claimant_clean))
check_duplicates(claimant_clean, ["lsoa_code", "year"], "claimant")

# ADI crime: average total_crime_rate within (lsoa_code, year)
adi_crime_clean = dedupe_aggregate(
    adi_crime,
    group_cols=["lsoa_code", "year"],
    agg_map={"total_crime_rate": "mean"},
)
adi_crime_clean["total_crime_rate"] = adi_crime_clean["total_crime_rate"].round(2)

check_row_counts("adi_crime dedup", len(adi_crime), len(adi_crime_clean))
check_duplicates(adi_crime_clean, ["lsoa_code", "year"], "adi_crime")

# Health: average total_prevalence_rate within (lsoa_code, year)
health_clean = dedupe_aggregate(
    health,
    group_cols=["lsoa_code", "year"],
    agg_map={"total_prevalence_rate": "mean"},
)
health_clean["total_prevalence_rate"] = health_clean["total_prevalence_rate"].round(2)

check_row_counts("health dedup", len(health), len(health_clean))
check_duplicates(health_clean, ["lsoa_code", "year"], "health")

### 5f. Write per-metric parquets to silver

In [0]:
write_parquet(claimant_clean,  f"{SILVER_DIR}/claimant_clean.parquet")
write_parquet(adi_crime_clean, f"{SILVER_DIR}/adi_crime_clean.parquet")
write_parquet(health_clean,    f"{SILVER_DIR}/health_clean.parquet")

---
## Dataset 3: House Prices (UK Land Registry PPD)

- **Source:** bronze Delta table `bronze_price_paid` (all years, all string columns)
- **Reporting grain:** Transaction-level (one row per unique transaction)
- **Output:** `{SILVER_DIR}/houseprices_clean.parquet`
- **Columns:** `date_of_transfer`, `postcode`, `property_type`, `ppd_category_type`, `price`

Cleaning rules per `documentation/houseprices_preprocess.md`:
- Residential property types only: D (Detached), S (Semi-detached), T (Terraced), F (Flat)
- Standard sales only: `ppd_category_type == 'A'`
- Realistic prices only: `price >= 50,000`

### 6a. Prepare function (constants live in Section 1)

In [0]:
def prepare_price_paid(df):
    """Clean one raw per-file PPD frame. Returns (cleaned_df, stats_dict).

    stats_dict records row counts after each filter so per-filter attribution
    can be reported once all files are processed.
    """
    stats = {"raw": len(df)}

    df = df[HP_OUTPUT_COLUMNS].copy()
    df["price"]            = pd.to_numeric(df["price"], errors="coerce")
    df["date_of_transfer"] = pd.to_datetime(df["date_of_transfer"], errors="coerce")
    df["postcode"]         = standardise_postcode(df["postcode"])

    df = df[df["postcode"].notna() & (df["postcode"] != "")]
    stats["after_postcode_clean"] = len(df)

    df = df[df["property_type"].isin(RESIDENTIAL)]
    stats["after_residential_filter"] = len(df)

    df = df[df["ppd_category_type"] == "A"]
    stats["after_ppd_category_filter"] = len(df)

    df = df[df["price"] >= 50_000]
    stats["after_price_filter"] = len(df)

    return df, stats

### 6b. Load from bronze Delta table and clean per source file

In [0]:
hp_raw = read_bronze("bronze_price_paid")

cleaned, all_stats = [], []
for src_file, group_df in hp_raw.groupby("_source_file"):
    df, stats = prepare_price_paid(group_df)
    stats["file"] = os.path.basename(src_file)
    cleaned.append(df)
    all_stats.append(stats)
    print(f"  {stats['file']}: {stats['raw']:,} raw -> {stats['after_price_filter']:,} clean")

hp_dfs, hp_stats = cleaned, all_stats

### 6c. Per-filter attribution across all files

In [0]:
stats_df = pd.DataFrame(hp_stats).set_index("file")
totals   = stats_df.sum(numeric_only=True)

print("Filter attribution (all files combined):")
check_row_counts("  blank/null postcode filter",   int(totals["raw"]),                       int(totals["after_postcode_clean"]))
check_row_counts("  residential filter (D/S/T/F)", int(totals["after_postcode_clean"]),      int(totals["after_residential_filter"]))
check_row_counts("  ppd_category_type == 'A'",     int(totals["after_residential_filter"]),  int(totals["after_ppd_category_filter"]))
check_row_counts("  price >= 50,000",              int(totals["after_ppd_category_filter"]), int(totals["after_price_filter"]))

print("\nPer-file detail:")
display(stats_df)

### 6d. Concat + quality assessment

In [0]:
hp = pd.concat(hp_dfs, ignore_index=True)
assess_quality(hp, "house prices post-concat")

### 6e. Deduplication on transaction_id

In [0]:
before_dedup = len(hp)
hp = hp.drop_duplicates(subset="transaction_id", keep="last")
hp = hp.drop(columns="transaction_id").reset_index(drop=True)

check_row_counts("transaction_id dedup", before_dedup, len(hp))

### 6f. Category validation

In [0]:
check_categories(hp, "property_type",     expected=RESIDENTIAL)
check_categories(hp, "ppd_category_type", expected={"A"})

### 6g. Price & date sanity checks

In [0]:
print(f"Price -- min: {hp['price'].min():,}   max: {hp['price'].max():,}")
print(f"Date  -- min: {hp['date_of_transfer'].min().date()}   max: {hp['date_of_transfer'].max().date()}")
print(f"Columns: {list(hp.columns)}")
display(hp.head())

### 6h. Write parquet

In [0]:
write_parquet(hp, f"{SILVER_DIR}/houseprices_clean.parquet")

---
## Dataset 4: Postcode → LSOA Lookup

- **Source:** bronze Delta table `bronze_postcode_lookup` — two columns: `pcds`, `lsoa11`
- **Reporting grain:** Postcode (one row per unique postcode)
- **Output:** `{SILVER_DIR}/postcode_clean.parquet`
- **Columns:** `postcode`, `lsoa_code`

The ingestion layer already projected to only these two columns (from the 51-column ONSPD CSV).

### 7a. Load (two columns only)

In [0]:
pc = read_bronze("bronze_postcode_lookup")[["pcds", "lsoa11"]].copy()
pc = standardise_columns(pc, rename_map={"pcds": "postcode", "lsoa11": "lsoa_code"})

raw_pc_count = len(pc)
print(f"Raw rows: {raw_pc_count:,}")
display(pc.head())

### 7b. Clean -- trim whitespace, drop null/blank, deduplicate on postcode

In [0]:
pc["postcode"]  = pc["postcode"].str.strip()
pc["lsoa_code"] = pc["lsoa_code"].str.strip()

pc = pc.dropna(subset=["postcode", "lsoa_code"])
pc = pc[(pc["postcode"] != "") & (pc["lsoa_code"] != "")]
check_row_counts("drop null/blank rows", raw_pc_count, len(pc))

before_dedup = len(pc)
pc = pc.drop_duplicates(subset="postcode", keep="first").reset_index(drop=True)
check_row_counts("deduplicate on postcode", before_dedup, len(pc))

### 7c. Validate

In [0]:
assess_quality(pc, "postcode_clean")
check_duplicates(pc, ["postcode"], "postcode_clean")
display(pc.head())

### 7d. Write parquet

In [0]:
write_parquet(pc, f"{SILVER_DIR}/postcode_clean.parquet")

---
## 8. Final Validation Summary

Read every silver parquet back and assert grain / category / range invariants. This is the single end-of-notebook check that the pipeline produced what downstream stages expect.

In [0]:
outputs = {
    "claimant_clean.parquet":    f"{SILVER_DIR}/claimant_clean.parquet",
    "adi_crime_clean.parquet":   f"{SILVER_DIR}/adi_crime_clean.parquet",
    "health_clean.parquet":      f"{SILVER_DIR}/health_clean.parquet",
    "houseprices_clean.parquet": f"{SILVER_DIR}/houseprices_clean.parquet",
    "postcode_clean.parquet":    f"{SILVER_DIR}/postcode_clean.parquet",
}
# Per-force police crime parquets (one entry per POLICE_FORCES element).
for force in POLICE_FORCES:
    name = f"{force}_crime_clean.parquet"
    outputs[name] = f"{SILVER_DIR}/{name}"

print("=== Silver layer outputs ===")
loaded = {}
missing = []
for name, path in outputs.items():
    if not os.path.exists(path):
        print(f"  {name}: MISSING (skipped) - {path}")
        missing.append(name)
        continue
    df_check = pd.read_parquet(path)
    size_mb  = os.path.getsize(path) / 1_048_576
    loaded[name] = df_check
    print(f"  {name}: shape={df_check.shape}  size={size_mb:.1f} MB  cols={list(df_check.columns)}")

# ── Invariants (each guarded so missing parquets don't block other checks) ──

# Police crime: each per-force file should only contain its own force, and
# (crime_id, lsoa_code) must be unique among rows with a non-null crime_id.
for force in POLICE_FORCES:
    name = f"{force}_crime_clean.parquet"
    if name not in loaded:
        continue
    df_force = loaded[name]
    assert (df_force["force_name"] == force).all(), \
        f"{name}: rows contain a non-{force} force_name"
    assert df_force.loc[df_force["crime_id"].notna(), ["crime_id", "lsoa_code"]].duplicated().sum() == 0, \
        f"{name}: duplicates at (crime_id, lsoa_code)"

if "claimant_clean.parquet" in loaded:
    assert loaded["claimant_clean.parquet"][["lsoa_code", "year"]].duplicated().sum() == 0, \
        "Claimant: duplicates at grain"

if "adi_crime_clean.parquet" in loaded:
    assert loaded["adi_crime_clean.parquet"][["lsoa_code", "year"]].duplicated().sum() == 0, \
        "ADI crime: duplicates at grain"

if "health_clean.parquet" in loaded:
    assert loaded["health_clean.parquet"][["lsoa_code", "year"]].duplicated().sum() == 0, \
        "Health: duplicates at grain"

if "houseprices_clean.parquet" in loaded:
    hp_chk = loaded["houseprices_clean.parquet"]
    assert set(hp_chk["property_type"].unique()) <= RESIDENTIAL,  "HP: non-residential type"
    assert set(hp_chk["ppd_category_type"].unique()) == {"A"},    "HP: non-A ppd_category"
    assert hp_chk["price"].min() >= 50_000,                       "HP: price below 50k"

if "postcode_clean.parquet" in loaded:
    assert loaded["postcode_clean.parquet"]["postcode"].is_unique, "Postcode: non-unique"

print(f"\nValidated {len(loaded)}/{len(outputs)} silver outputs.")
if missing:
    print(f"Skipped (file not found): {missing}")
print("Note: adi_transformed.parquet and houseprices_transformed.parquet are produced downstream by 03_feature_engineering_and_transformation.")